<center><h2><span style="font-weight:bolder; color:olivedrab; font-size:120%">Logistic Regression Project: Diabetes Prediction</span></h2></center>

<center><h2><span style="font-weight:bolder; color:black; font-size:90%">Melissa Jalali Monfared</span></h2></center>

<a id="content"></a>    
<div style="border-radius:20px; padding: 15px; font-size:110%; text-align:left">

<center><h2><span style="font-weight:bolder; color:black; font-size:70%">       Table of Contents:</span></h2></center>

 *  **[- | Introduction](#in)**
 *  **[- | About Dataset](#about)**
 *  **[- | PreProcessing & Visualization](#pre)**
 *  **[- | ML: Logistic Regression](#ml)**

<a id="in"></a>
# <p style="background-color:darkseagreen;font-family:newtimeroman;font-size:100%;color:blackData Description ;text-align:center;border-squar:15px 50px; padding:7px">Introduction</p>

### Diabetes, often called "sugar" but not containing sugar itself, is a chronic condition where blood sugar levels (glucose) are consistently higher than normal. It's one of the most common diseases globally and can affect anyone, regardless of age or location.With machine learning models, we can predict diabetes and we must do our best to increase the accuracy of the model and not endanger the health of the patients with incorrect predictions. One of the models that can be used for this prediction is logistic regression.

<a id="about"></a>
# <p style="background-color:darkseagreen;font-family:newtimeroman;font-size:100%;color:blackData Description ;text-align:center;border-squar:15px 50px; padding:7px">About Dataset</p>

### This dataset consists of 768 observations & 8 numerical independent variables.
#### Dependent and target variable is OUTCOME. **1** means diabetes test result being positive, **0** means indicates negative.
* **Pregnancies**: Number of Times Being Pregnant
* **Glucose**: Plasma Glucose Concentration (a 2 hours in an oral glucose tolerance test)
* **BloodPressure**: Diastolic Blood Pressure (mm Hg)
* **SkinThickness**: Triceps Skin Fold Thickness (mm)
* **Insulin**: 2-Hour Serum Insulin (mu U/ml)
* **BMI**: Body Mass Index (weight in kg/(height in m)^2)
* **DiabetesPedigreeFunction**: Diabetes Pedigree Function
* **Age**: Age
* **Outcome**: Class variable ( 0 - 1)

<a id="pre"></a>
# <p style="background-color:darkseagreen;font-family:newtimeroman;font-size:100%;color:blackData Description ;text-align:center;border-squar:15px 50px; padding:7px">PreProcessing & Visualization</p>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_validate
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)

In [ ]:
data = pd.read_csv("/kaggle/input/diabetes/diabetes.csv")
df = data.copy()

In [ ]:
def check_df(dataframe: object, head: object = 5) -> object:
    print("Shape")
    print(dataframe.shape)
    print("Types")
    print(dataframe.dtypes)
    print("NANs")
    print(dataframe.isnull().sum())
    print("Quantiles")
    print(dataframe.quantile([0, 0.05,0.1, 0.25, 0.50,0.75, 0.90, 0.95, 0.99, 1]).T)
check_df(df)

**there is no NAN data & no object variable type**

In [ ]:
df.describe(include='all')

**we can see statistical information on the table above**

In [ ]:
df.info()

In [ ]:
df.groupby(['Outcome']).agg({"Age":["mean", "median"],
                            "Glucose":["mean","median"],
                            "Insulin":["mean", "median"],
                            "DiabetesPedigreeFunction":["mean", "median"],
                            "Pregnancies":["mean", "median"],
                            "BMI":["mean", "median"],
                            "SkinThickness":["mean", "median"],
                            "BloodPressure":["mean", "median"]}) 

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    # cat_cols, cat_but_car
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]
    return cat_cols, num_cols, cat_but_car
cat_cols, num_cols, cat_but_car = grab_col_names(df)

In [ ]:
def cat_summary(dataframe, col_name, plot=False):
    print(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        "Ratio": 100 * dataframe[col_name].value_counts() / len(dataframe)}))
    if plot:
        sns.set_style("darkgrid")
        fig, ax = plt.subplots(1, 2)
        ax = np.reshape(ax, (1, 2))
        ax[0, 0] = sns.countplot(x=dataframe[col_name], color="mediumaquamarine", ax=ax[0, 0])
        ax[0, 0].set_ylabel('Count')
        ax[0, 0].set_xticklabels(ax[0, 0].get_xticklabels(), rotation=-45)
        ax[0, 1] = plt.pie(dataframe[col_name].value_counts().values, labels=dataframe[col_name].value_counts().keys(),
                           colors=sns.color_palette('crest'), autopct='%.0f%%')
        plt.title("Percentage")
        fig.set_size_inches(12, 6)
        fig.suptitle('Analysis of Categorical Variables', fontsize=14)
        plt.show()
for col in cat_cols:
    cat_summary(df, col, plot=True)

**analysis of categorical variables, count plot & pie plot**

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
# Suppress warnings to avoid clutter
warnings.filterwarnings("ignore", category=FutureWarning)
def num_summary(dataframe, numerical_col):
    fig, ax = plt.subplots(1, 2, figsize=(10, 6))  # Specify figsize for consistent visuals
    # Histogram
    sns.histplot(x=dataframe[numerical_col], color="darkseagreen", bins=20, ax=ax[0])
    ax[0].set_ylabel('Count')
    ax[0].set_title('Distribution')
    # Box plot
    sns.boxplot(y=dataframe[numerical_col], color="yellowgreen", showmeans=True, ax=ax[1])  # Add mean indicator
    ax[1].set_title('BOX PLOTS')
    fig.suptitle('Analysis of Numerical Variables', fontsize=14)
    plt.tight_layout()  # Adjust spacing for better readability
    plt.show()
for col in df[num_cols]:
    num_summary(df, col)

**analysis of numerical variables, count plot & box plot**

In [ ]:
def correlated_map(dataframe, plot=False):
    corr = dataframe.corr()
    if plot:
        sns.set(rc={'figure.figsize': (16, 8)})
        sns.heatmap(corr, cmap="GnBu", annot=True, linewidths=.6)
        plt.xticks(rotation=60, size=10)
        plt.yticks(size=10)
        plt.title('Analysis of Correlations', size=14)
        plt.show()
correlated_map(df, plot=True)

**we can see the correlations from above, highest correlations are between Age&Pregnancies, Outcome&Glucose, SkinThickness&Insulin**
**Glucose have the highest correlation with outcome which is our target**
**after that, BMI, Age & Pregnancies have the highest correlation with outcome in comparison with other features**

In [ ]:
def outlier_thresholds(dataframe, col_name, q1=0.05, q3=0.95):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit
for col in num_cols:
    print(outlier_thresholds(df, col))

**In the above way, we define a limit for the noises & outliers**

In [ ]:
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit
for col in num_cols:
    replace_with_thresholds(df,col)

**we replace noises & outliers with defined thresholds**

In [ ]:
df.describe().T

In [ ]:
dff=df.copy()

In [ ]:
# labeling bmi based on body mass index (BMI) informations
bmi_labels = ["UnderWeight", "NormalWeight", "Overweight", "ObesityLVL1", "ObesityLVL2", "ObesityLVL3"]
dff['BMI_Cat'] = pd.cut(dff['BMI'], [-1, 18.5, 25, 30, 35, 40, dff['BMI'].max()], 
                       labels=bmi_labels)
cat_summary(dff,"BMI_Cat", plot=True)

**with the help of data available in medical science, we have classified our dataset as above**
**& we can see that how many of our datas, are in which categories.**
**UnderWeight Count < ObesityLVL3 Count < NormalWeight Count < ObesityLVL2 Count < Overweight Count < ObesityLVL3 Count**

In [ ]:
dff["New_Age_Cat"] = dff["Age"].apply(lambda x: "Young" if x < 30 else ("Mature" if 30 <= x <= 50 else "Senior"))
cat_summary(dff,"New_Age_Cat", plot=True)

**we have classified our dataset as above to young age, mature & senior**
**& we can see that how many of our datas, are in which categories.**
**Senior Count < Mature Count < Young Count**

In [ ]:
dff["GlucoseCat"] = dff["Glucose"].apply(lambda x: "Normal" if x < 140 else ("Prediabetes" if 140 <= x <= 200 else "diabetes"))
cat_summary(dff,"GlucoseCat", plot=True)

**A blood sugar level less than 140 mg/dL (7.8 mmol/L) is normal.**
**A reading of more than 200 mg/dL (11.1 mmol/L) after two hours indicates diabetes.**
**A reading between 140 and 199 mg/dL (7.8 mmol/L and 11.0 mmol/L) indicates prediabetes.**
**with the help of data available in medical science, we have classified our dataset as above**
**& we can see that how many of our datas, are in which categories.**
**Prediabetes Count < Normal Count**

In [ ]:
# labeling based on medical informations available
bp_labels = ["Optimal", "Normal", "HighNormal", "Grade1_Hypertension", "Grade2_Hypertension", "Grade3_Hypertension"]
dff['BloodPressureCat'] = pd.cut(dff['BloodPressure'], [-1, 80,  85, 90, 100, 110, dff['BloodPressure'].max()], labels=bp_labels)
cat_summary(dff,"BloodPressureCat", plot=True)

**with the help of data available in medical science, we have classified our dataset as above**
**& we can see that how many of our datas, are in which categories.**
**Grade3_Hypertension Count < Grade2_Hypertension Count < Normal Count < HighNormal Count < Optimal Count**

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Glucose"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between Glucose & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 205 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Glucose', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Pregnancies"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between Pregnancies & Diabetes (the density is visible)", fontsize = 20)
plt.xticks (range (0 , 30 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Pregnancies', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BloodPressure"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between BloodPressure & Diabetes (the density is visible)", fontsize = 20)
plt.xticks (range (0 , 140 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BloodPressure', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["SkinThickness"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between SkinThickness & Diabetes (the density is visible)", fontsize = 20)
plt.xticks (range (0 , 120 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('SkinThickness', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Insulin"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between Insulin & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 1000 , 100), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Insulin', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BMI"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between BMI & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 70 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BMI', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Age"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between Age & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 100 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Age', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["DiabetesPedigreeFunction"] , df["Outcome"] , color = "darkseagreen")
plt.title ("Relationship between DiabetesPedigreeFunction & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 2), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('DiabetesPedigreeFunction', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
columns1 = ['Pregnancies','Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
df_mean = df.groupby('Outcome')[columns1].mean()
df_mean.plot(kind='barh', stacked=True, figsize=(15, 10), cmap='crest')
color_list = ['red', 'orange', 'yellow', 'green', 'blue', 'darkblue', 'pink', 'teal', 'gray']
plt.xlabel('Average')
plt.title('We can see the average of each Feature VS each Outcome')
plt.legend(loc='lower right')
plt.show()

In [ ]:
n = df.groupby('Outcome')[['Pregnancies','Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']].mean()
n.plot(kind='bar', figsize=(15, 10), cmap='crest')
plt.title("We can see the influence of each column on Outcome")
plt.xlabel('Outcome')
plt.ylabel('Average')
plt.show()

In [ ]:
columns = ['Pregnancies','Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
for column in columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[column], color='darkseagreen')
    plt.title(column)
    stats = df[column].describe()
    stats_text = "\n".join([f"{key}: {value:.2f}" for key, value in stats.items()])
    print(f"\n{column} Statistics:\n{stats_text}\n")
    plt.show()

In [ ]:
columns = ['Pregnancies','Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
for column in columns:
    plt.figure(figsize=(12,6))
    sns.violinplot(x=df[column], color='darkseagreen')
    plt.title(column)
    plt.show()

**the data distribution can be seen in the above plots**

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(data['DiabetesPedigreeFunction'].dropna(),kde=True)

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='Pregnancies',data=data, palette='crest')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Pregnancies',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Pregnancies',fontsize=30)
plt.grid()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(data['Pregnancies'].dropna(),kde=True)

In [ ]:
plt.figure(figsize=(16,10))
sns.countplot(x="Pregnancies",data=data,hue="Outcome", palette='crest')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='Glucose',data=data, palette='crest')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Glucose',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Glucose',fontsize=30)
plt.grid()

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(data['Glucose'].dropna(),kde=True)

In [ ]:
glucose_bins=pd.cut(df["Glucose"],bins=[40,90,130,200],labels=["40-90","90-130","130-200"])
plt.figure(figsize=(20,10))
sns.countplot(x=glucose_bins,data=data,hue="Outcome",palette='crest')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='BloodPressure',data=data,palette='crest')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('BloodPressure',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of BloodPressure',fontsize=30)
plt.grid()

In [ ]:
warnings.filterwarnings("ignore")
sns.distplot(data['BloodPressure'].dropna(),kde=True)

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="BloodPressure",data=data,hue="Outcome",palette='crest')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='SkinThickness',data=data,palette='crest')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('SkinThickness',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of SkinThickness',fontsize=30)
plt.grid()

In [ ]:
warnings.filterwarnings("ignore")
sns.distplot(data['SkinThickness'].dropna(),kde=True)

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="SkinThickness",data=data,hue="Outcome",palette='crest')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='Age',data=data,palette='crest')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Age',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Age',fontsize=30)
plt.grid()

In [ ]:
warnings.filterwarnings("ignore")
sns.distplot(data['Age'].dropna(),kde=True)

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="Age",data=data,hue="Outcome",palette='crest')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='BMI', data=df, palette='viridis')

**the dispersion of BMI at each age can be seen above**

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='SkinThickness', data=df, palette='viridis')

**the dispersion of SkinThickness at each age can be seen above**

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='BloodPressure', data=df, palette='viridis')

**the dispersion of BloodPressure at each age can be seen above**

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='Glucose', data=df, palette='viridis')

**the dispersion of Glucose at each age can be seen above**

In [ ]:
sns.pairplot(data=df, diag_kind='kde', hue='Outcome')
plt.show()

In [ ]:
fig = plt.figure(figsize=(15, 8.5),dpi=100)
ax = fig.add_subplot(111,projection='3d')
p1 = ax.scatter(df['BMI'], df['Age'], df['Glucose'],c=df['Outcome'],cmap='crest')
fig.colorbar(p1, shrink=0.3,label='Outcome',anchor=(3,1))
ax.set_xlabel("BMI")
ax.set_ylabel("Age")
ax.set_zlabel("Glucose")
ax.set_title("Correlation Between Glucose & Age & BMI",fontdict={'fontsize': 12})
ax.patch.set_facecolor("white")
fig.show()

In [ ]:
fig = plt.figure(figsize=(15, 8.5),dpi=100)
ax = fig.add_subplot(111,projection='3d')
p1 = ax.scatter(df['BloodPressure'], df['SkinThickness'], df['Glucose'],c=df['Outcome'],cmap='cividis')
fig.colorbar(p1, shrink=0.3,label='Outcome',anchor=(3,1))
ax.set_xlabel("BloodPressure")
ax.set_ylabel("SkinThickness")
ax.set_zlabel("Insulin")
ax.set_title("Correlation Between Insulin & SkinThickness & BloodPressure",fontdict={'fontsize': 12})
ax.patch.set_facecolor("white")
fig.show()

In [ ]:
# Label Encoding
def label_encoder(dataframe, binary_col):
    labelencoder = LabelEncoder()
    dataframe[binary_col] = labelencoder.fit_transform(dataframe[binary_col])
    return dataframe
binary_cols = [col for col in df.columns if df[col].dtype not in [int, float] and df[col].nunique() == 2]
len(binary_cols)

In [ ]:
for col in binary_cols:
    label_encoder(df, col)
df.head()

In [ ]:
# One-Hot Encoding
def one_hot_encoder(dataframe, categorical_cols, drop_first=False):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe

ohe_cols = [col for col in df.columns if 10 >= df[col].nunique() > 2]
df = one_hot_encoder(df, ohe_cols, drop_first=True)
df.head()

<a id="ml"></a>
# <p style="background-color:darkseagreen;font-family:newtimeroman;font-size:100%;color:blackData Description ;text-align:center;border-squar:15px 50px; padding:7px">ML: Logistic Regression</p>

In [ ]:
from sklearn import preprocessing
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
normal = scaler.fit_transform(df)
df2 = pd.DataFrame(normal, columns=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'])

In [ ]:
x = pd.DataFrame(df2, columns=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'])
y = df2.Outcome.values.reshape(-1,1)

In [ ]:
# create train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [ ]:
# create model
logreg = LogisticRegression(solver='liblinear')

In [ ]:
# cross validation
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
kf = KFold(10)
result_validation = cross_val_score(logreg, x_train, y_train, cv=kf)

In [ ]:
result_validation

In [ ]:
print(np.mean(result_validation))

In [ ]:
# fit model on dataset train
logreg.fit(x_train, y_train.ravel())

In [ ]:
# prediction
y_pred = logreg.predict(x_test)
y_pred

In [ ]:
# compare peredict and actual
compare =pd.DataFrame({'actual': y_test.flatten(),
          'predict' : y_pred.flatten()})
compare

In [ ]:
from sklearn import metrics
print("Accuracy : ",metrics.accuracy_score(y_test, y_pred))

In [ ]:
fpr, tpr, _ = metrics.roc_curve(y_test, y_pred)
plt.plot(fpr, tpr, label='data1', color="seagreen")
plt.legend(loc=4)
plt.show()

In [ ]:
y_pred_proba = logreg.predict_proba(x_test)[ : : ,1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label='data1', color="seagreen")
plt.legend(loc='best')
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
# class
logreg.classes_

In [ ]:
# coef
logreg.coef_

In [ ]:
# intercept
logreg.intercept_

In [ ]:
# predict
logreg.predict(x)

In [ ]:
# predict_proba
logreg.predict_proba(x)

In [ ]:
# score
logreg.score(x, y)

In [ ]:
# confusion matrix
confusion_matrix(y, logreg.predict(x))

In [ ]:
# confusion matrix heatmap
plt.figure(figsize=(10, 6))
sns.set(font_scale=1.2)
sns.heatmap(confusion_matrix(y, logreg.predict(x)) , annot=True, fmt="d", cmap="YlGn", cbar=False,
            xticklabels=['Predict 0s', 'Predict 1s'], yticklabels=['Actual 0s', 'Actual 1s'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 453 people did not have diabetes and it was predicted that they did not have diabetes
## 138 data had diabetes and it was predicted that they had diabetes
## 47 people did not have diabetes and it was predicted that they had diabetes
## 130 people had diabetes and it was predicted that they did not have diabetes

In [ ]:
print(classification_report(y, logreg.predict(x)))

In [ ]:
feature_imp = pd.DataFrame({'Value': logreg.coef_[0], 'Feature': x.columns})
plt.figure(figsize=(16, 8))
sns.set(font_scale=1)
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",
                                                                     ascending=False)[0:8], palette="crest")
plt.title('Features')
plt.tight_layout()
plt.show()

In [ ]:
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
shapes = {
    'X_train': x_train.shape[0],
    'y_train': y_train.shape[0],
    'X_test': x_test.shape[0],
    'y_test': y_test.shape[0]}
plt.figure(figsize=(16, 6))
plt.bar(shapes.keys(), shapes.values(),color="darkseagreen")
plt.ylabel('Count')
plt.title('Distribution of Training & Validation Sets')
plt.show()

In [ ]:
from numpy import reshape
y = reshape(y, (y.shape[0],))

In [ ]:
from numpy import ravel
y = ravel(y)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, RepeatedStratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.validation import column_or_1d 
scaler = MinMaxScaler(feature_range=(0, 1))
normal = scaler.fit_transform(df)
df10 = pd.DataFrame(normal, columns=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'])
x = pd.DataFrame(df10, columns=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'])
y = df10.Outcome.values
y = column_or_1d(y, warn=True)  
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
model1 = LogisticRegression()
solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
penalty = ['l2']
c_values = [100, 10, 1.0, 0.1, 0.01]
kf = KFold(10)
result_validation = cross_val_score(model1, x_train, y_train, cv=kf)
model1.fit(x_train, y_train.ravel())
y_pred = model1.predict(x_test)
grid = dict(solver=solvers,penalty=penalty,C=c_values)
cv = RepeatedStratifiedKFold(n_splits=10, random_state=0)
grid_search = GridSearchCV(estimator=model1, param_grid=grid, n_jobs=-1, cv=cv, scoring='accuracy',error_score=0)
grid_result = grid_search.fit(x_train, y_train) 
print(f"Best: {grid_result.best_score_} by using {grid_result.best_params_}")
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print(f"{mean} ({stdev}) with: {param}")
     

**we created a model by using C=100, penalty=l2 & slover=liblinear (these changes gives us a better score based on best_score_ that we used on above code)**

In [ ]:
# predict
model1.predict(x)

In [ ]:
# Create a confusion matrix heatmap
plt.figure(figsize=(10, 6))
sns.set(font_scale=1.2)
sns.heatmap(confusion_matrix(y, model1.predict(x)) , annot=True, fmt="d", cmap="YlGn", cbar=False,
            xticklabels=['Predict 0s', 'Predict 1s'], yticklabels=['Actual 0s', 'Actual 1s'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 451 people did not have diabetes and it was predicted that they did not have diabetes
## 141 data had diabetes and it was predicted that they had diabetes
## 49 people did not have diabetes and it was predicted that they had diabetes
## 127 people had diabetes and it was predicted that they did not have diabetes
# It can be seen that the number of destructive and dangerous forecasts has decreased by 3, so with the changes applied in the parameters, we had a more accurate forecast.

In [ ]:
print(classification_report(y, model1.predict(x)))

In [ ]:
feature_imp = pd.DataFrame({'Value': model1.coef_[0], 'Feature': x.columns})
plt.figure(figsize=(16, 8))
sns.set(font_scale=1)
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",
                                                                     ascending=False)[0:8], palette="crest")
plt.title('Features')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = metrics.roc_curve(y_test, y_pred)
plt.plot(fpr, tpr, label='data1', color="seagreen")
plt.legend(loc=4)
plt.show()

In [ ]:
y_pred_proba = model1.predict_proba(x_test)[ : : ,1]
fpr, tpr, _ = metrics.roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label='data1', color="seagreen")
plt.legend(loc='best')
plt.show()